In [2]:
import string
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

# Ensure required NLTK tokenizers and datasets are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# ==========================================
# 1. FAQ DATASET COLLECTION
# ==========================================
faq_data = [
    {
        "question": "How can I track my order?",
        "answer": "You can track your order by logging into your account and navigating to the 'My Orders' section."
    },
    {
        "question": "What is your return policy?",
        "answer": "We offer a 30-day return policy for unused items in their original packaging."
    },
    {
        "question": "What payment methods do you accept?",
        "answer": "We accept Credit/Debit Cards, PayPal, Apple Pay, and Google Pay."
    },
    {
        "question": "How long does shipping take?",
        "answer": "Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days."
    },
    {
        "question": "How do I reset my password?",
        "answer": "Click on 'Forgot Password' on the login screen and follow the instructions sent to your email."
    },
    {
        "question": "How do I contact customer support?",
        "answer": "You can reach us via email at support@example.com or call our toll-free number at 1-800-555-0199."
    }
]

# Convert dataset to Pandas DataFrame
df = pd.DataFrame(faq_data)

# ==========================================
# 2. NLP PREPROCESSING ROUTINE
# ==========================================
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text: str) -> str:
    """
    Applies NLP cleaning:
    - Lowercasing
    - Tokenization
    - Removal of punctuation & stop words
    - Lemmatization
    """
    # Lowercase
    text = text.lower()

    # Tokenize
    tokens = word_tokenize(text)

    # Clean & Lemmatize
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in string.punctuation and token not in stop_words
    ]

    return " ".join(cleaned_tokens)

# Preprocess all FAQ questions in advance
df['processed_question'] = df['question'].apply(preprocess_text)

# ==========================================
# 3. TF-IDF VECTORIZATION
# ==========================================
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['processed_question'])

# ==========================================
# 4. SIMILARITY MATCHING ENGINE
# ==========================================
def get_bot_response(user_query: str, threshold: float = 0.2) -> str:
    """
    Matches the user question with the FAQ dataset using Cosine Similarity.
    Returns fallback message if similarity score is below the threshold.
    """
    # Preprocess user query
    processed_query = preprocess_text(user_query)

    # If query becomes empty after removing stop words
    if not processed_query.strip():
        return "Please ask a specific question so I can help you better."

    # Vectorize user query using the trained vectorizer
    user_vector = vectorizer.transform([processed_query])

    # Calculate Cosine Similarity against all FAQs
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)[0]

    # Find index of best matching question
    best_match_idx = similarity_scores.argmax()
    best_score = similarity_scores[best_match_idx]

    # Check threshold for out-of-scope questions
    if best_score < threshold:
        return "I'm sorry, I couldn't find an answer to your question. Please try rephrasing or contact support at support@example.com."

    return df.iloc[best_match_idx]['answer']

# ==========================================
# 5. GRADIO CHAT UI
# ==========================================
def chat_handler(message, history):
    return get_bot_response(message)

app = gr.ChatInterface(
    fn=chat_handler,
    title="🤖 E-Commerce FAQ Chatbot",
    description="Ask me about orders, shipping, payment methods, or returns!",
)

if __name__ == "__main__":
    app.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
